In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("https://arxiv.org/pdf/1706.03762")
docs = loader.load()

/tmp/ipykernel_7355/870890616.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(docs)


In [1]:
from google import genai

client = genai.Client(api_key='')

result = client.models.embed_content(
        model="gemini-embedding-2",
        contents=texts[0].page_content
)

print(result.embeddings)


ValueError: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [6]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="chroma_db"
)

In [19]:
'''vector_store.add_documents(texts)

print("Documents added successfully!")'''

'vector_store.add_documents(texts)\n\nprint("Documents added successfully!")'

In [20]:
retriever=vector_store.as_retriever()

In [21]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions based on the provided context.

Previous conversation:
{chat_history}

Context:
{context}

Current question:
{question}

Answer the question using the context and previous conversation.
If the answer cannot be found in the context, say that you don't know.

Answer:
""")

In [9]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [10]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="poolside/laguna-s-2.1:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER"],
)

In [94]:
# %%
def format_history(messages):

    if not messages:
        return "No previous conversation."

    return "\n".join(
        f"{message.type}: {message.content}"
        for message in messages
    )

In [88]:
from langchain_core.output_parsers import StrOutputParser

In [95]:

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": RunnableLambda(
            lambda x: format_docs(
                retriever.invoke(
                    x["question"]
                )
            )
        ),

        "question": RunnableLambda(
            lambda x: x["question"]
        ),

        "chat_history": RunnableLambda(
            lambda x: x["chat_history"]
        )
    }

    | prompt
    | llm
    | StrOutputParser()
)

In [76]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

history = SQLChatMessageHistory(
    session_id="user_123",
    connection="sqlite:///chat_history.db"
)

In [77]:
from langchain_core.messages import HumanMessage, AIMessage

history.add_message(
    HumanMessage(question)
)

history.add_message(
    AIMessage(answer)
)

In [89]:
'''history = SQLChatMessageHistory(
    session_id="user_123",
    connection="sqlite:///chat_history.db"
)

for msg in history.messages:
    print(msg.content)'''

'history = SQLChatMessageHistory(\n    session_id="user_123",\n    connection="sqlite:///chat_history.db"\n)\n\nfor msg in history.messages:\n    print(msg.content)'

In [100]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage


DATABASE_URL = "sqlite:///chat_history.db"


def format_history(messages):

    if not messages:
        return "No previous conversation."

    return "\n".join(
        f"{message.type}: {message.content}"
        for message in messages
    )


def ask_rag(question, session_id="user_123"):

    # -----------------------------------
    # 1. Open the conversation history
    # -----------------------------------

    history = SQLChatMessageHistory(
        session_id=session_id,
        connection=DATABASE_URL
    )

    # -----------------------------------
    # 2. Get previous messages
    # -----------------------------------

    previous_messages = history.messages

    # -----------------------------------
    # 3. Convert history to text
    # -----------------------------------

    chat_history = format_history(
        previous_messages
    )

    # -----------------------------------
    # 4. Run RAG
    # -----------------------------------

    answer = rag_chain.invoke({
        "question": question,
        "chat_history": chat_history
    })

    # -----------------------------------
    # 5. Save user question
    # -----------------------------------

    history.add_message(
        HumanMessage(
            content=question
        )
    )

    # -----------------------------------
    # 6. Save AI answer
    # -----------------------------------

    history.add_message(
        AIMessage(
            content=answer
        )
    )

    # -----------------------------------
    # 7. Return answer
    # -----------------------------------

    return answer

In [120]:
answer = ask_rag(
    "What is attention mechanism?"
)

print(answer)

The attention mechanism is a function that maps a **query** and a set of **key-value pairs** to an output vector. The output is computed as a **weighted sum of the values**, where the weights correspond to the compatibility between the query and each key. This compatibility is determined by a function (e.g., a dot product of the query with a key), and the weights are typically normalized using a softmax function. 

In the context of the Transformer model, a specific implementation called **Scaled Dot-Product Attention** is used, which calculates the weights by taking the dot product of the query with all keys, scales them by \( \sqrt{d_k} \) (to prevent large magnitudes during softmax), and applies softmax before multiplying with the values. 

Mathematically, this is represented as:

\[
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

Here, \( Q \), \( K \), and \( V \) are matrices representing the packed queries, keys, and values, respectively. This

In [126]:

answer = ask_rag(
    "Why is it important?"
)

print(answer)

The attention mechanism is important for several reasons, as highlighted in the provided context and prior discussion:

1. **Eliminates Sequential Constraints**: Traditional recurrent neural networks (RNNs) require sequential processing of input sequences, which limits parallelization during training. This becomes computationally expensive and memory-intensive for long sequences. The Transformer, relying entirely on attention mechanisms instead of recurrence, **bypasses this sequential bottleneck**, enabling parallelization across all positions in the sequence. This allows faster training (e.g., reaching state-of-the-art results in 12 hours on 8 GPUs) and scales better with sequence length.

2. **Dynamic Global Dependencies**: Attention mechanisms allow the model to **directly model dependencies between distant elements** in the input and output sequences, regardless of their position. Unlike RNNs (with vanishing gradients) or convolutional networks (with limited receptive fields), att

In [123]:
# %%
def show_previous_chats(session_id="user_123"):

    history = SQLChatMessageHistory(
        session_id=session_id,
        connection=DATABASE_URL
    )

    messages = history.messages

    if not messages:
        print("No previous chats found.")
        return

    print("=" * 60)
    print(f"CHAT HISTORY: {session_id}")
    print("=" * 60)

    for message in messages:

        if message.type == "human":

            print("\n🧑 You:")
            print(message.content)

        elif message.type == "ai":

            print("\n🤖 AI:")
            print(message.content)

        print("-" * 60)

In [127]:

show_previous_chats()

CHAT HISTORY: user_123

🧑 You:
What is attention mechanism?
------------------------------------------------------------

🤖 AI:
The attention mechanism is a function that maps a **query** and a set of **key-value pairs** to an output vector. The output is computed as a **weighted sum of the values**, where the weights correspond to the compatibility between the query and each key. This compatibility is determined by a function (e.g., a dot product of the query with a key), and the weights are typically normalized using a softmax function. 

In the context of the Transformer model, a specific implementation called **Scaled Dot-Product Attention** is used, which calculates the weights by taking the dot product of the query with all keys, scales them by \( \sqrt{d_k} \) (to prevent large magnitudes during softmax), and applies softmax before multiplying with the values. 

Mathematically, this is represented as:

\[
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\ri

In [128]:
# %%
history = SQLChatMessageHistory(
    session_id="user_123",
    connection=DATABASE_URL
)

print("Number of messages:", len(history.messages))

for message in history.messages:
    print(
        f"{message.type}: {message.content}"
    )

Number of messages: 4
human: What is attention mechanism?
ai: The attention mechanism is a function that maps a **query** and a set of **key-value pairs** to an output vector. The output is computed as a **weighted sum of the values**, where the weights correspond to the compatibility between the query and each key. This compatibility is determined by a function (e.g., a dot product of the query with a key), and the weights are typically normalized using a softmax function. 

In the context of the Transformer model, a specific implementation called **Scaled Dot-Product Attention** is used, which calculates the weights by taking the dot product of the query with all keys, scales them by \( \sqrt{d_k} \) (to prevent large magnitudes during softmax), and applies softmax before multiplying with the values. 

Mathematically, this is represented as:

\[
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

Here, \( Q \), \( K \), and \( V \) are matrices represe

In [117]:
# %%
def clear_chat(session_id="user_123"):

    history = SQLChatMessageHistory(
        session_id=session_id,
        connection=DATABASE_URL
    )

    history.clear()

    print(
        f"Chat history cleared for: {session_id}"
    )

In [118]:
# %%
clear_chat()

Chat history cleared for: user_123
